# Phase 5 – RAG Assistant for Police Narrative Classification
**MAAI 902 – NLP & LLMs · Applied NLP Project**

A retrieval-augmented question-answering assistant that answers questions about the FIO data and the policy behind
its categories, using only approved documents and citing the passages it used. It also explains the classifier's
predictions by retrieving the Rule 323 definition of the predicted category.

| Document | Source | Role |
|---|---|---|
| **BPD Rule 323** – Field Interaction/Observation/Encounter Report (2 July 2015) | Boston Police Department, via the US Court of Appeals for the First Circuit | Official definitions of *Stopped*, *Encountered* and *Observed*, and documentation rules |
| **FIO Data Key (Mark43)** | City of Boston, Analyze Boston | Official field dictionary for the 2024 files |
| **Dataset card** | Written for this project from the Analyze Boston dataset page | Source, licence, systems, data notes |
| **Project model card** | Written for this project from the Phase 1–4 results | How the classifier was built, results, limitations, intended use |

**Pipeline:** documents → section-aware chunks → embeddings (`all-MiniLM-L6-v2`) in a FAISS index, plus BM25 →
hybrid retrieval → prompt with numbered sources → open-source LLM (`Qwen2.5-1.5B-Instruct`, runs locally in Colab,
no data leaves the session) → answer with citations, or a refusal when the documents do not cover the question.

**Runtime:** T4 GPU recommended (the LLM is slow on CPU). Takes about 10 minutes including model downloads.

## 0. Setup

In [1]:
!pip -q install faiss-cpu pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 10.2 MB/s eta 0:00:00


In [2]:
import os, re, json, glob, math, time
from collections import Counter
import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader

pd.set_option("display.max_colwidth", 160)
RAG_DIR = "rag_outputs"; os.makedirs(RAG_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (the LLM step will be slow)")

Device: Tesla T4


### 0.1 Upload the two official documents
Upload `rule323.pdf` and `fiokeymark43-1-1.xlsx` (any file names containing *323* and *key* work).

In [3]:
def find(pattern):
    hits = [f for f in glob.glob("*") + glob.glob("rag_docs/*") if re.search(pattern, f, re.I)]
    return hits[0] if hits else None

RULE_PATH, KEY_PATH = find(r"323.*\.pdf$"), find(r"key.*\.xlsx$")
if not (RULE_PATH and KEY_PATH):
    from google.colab import files
    files.upload()
    RULE_PATH, KEY_PATH = find(r"323.*\.pdf$"), find(r"key.*\.xlsx$")
print("Rule 323:", RULE_PATH, "| Data key:", KEY_PATH)

Saving fiokeymark43-1-1.xlsx to fiokeymark43-1-1.xlsx
Saving rule323.pdf to rule323.pdf
Rule 323: rule323.pdf | Data key: fiokeymark43-1-1.xlsx


## 1. Document preparation
### 1.1 BPD Rule 323 – cleaning
The court library's copy carries a watermark line ("Last Viewed by First Circuit Library…") and a repeated page
header on every page; both are removed. The rule is then split at its own structure – *Section N* and *Sec. N.N* –
so each definition is its own chunk and can be cited precisely (e.g. *Rule 323 §3.1*).

In [4]:
def load_rule323(path):
    text = "\n".join(p.extract_text() for p in PdfReader(path).pages)
    text = re.sub(r"Last Viewed by First Circuit Library on \d{2}/\d{2}/\d{4}", " ", text)
    text = re.sub(r"Boston Police Department Rules and Procedures\s+Rule 323\s+July 02, 2015", " ", text)
    text = re.sub(r"Page \d+ of \d+", " ", text)
    text = re.sub(r"[\uf0b7\uf0a7\u2022\uf076]", " • ", text)          # PDF bullet glyphs
    text = re.sub(r"\b([A-Z])\s*\n\s*([a-z])", r"\1\2", text)          # "P\nolice" -> "Police"
    text = re.sub(r"\bth ose\b", "those", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"Section (\d+) :", r"Section \1:", text)          # "Section 7\n:" in the PDF
    text = re.sub(r"(?<![\w'’])([b-hj-z]) ([a-z]{2,})\b", r"\1\2", text)   # "o bservation" -> "observation"
    text = text.replace(" —", " —").replace("’", "'").replace("“", '"').replace("”", '"')
    return text

rule_text = load_rule323(RULE_PATH)
print(len(rule_text.split()), "words after cleaning")
print(rule_text[:400])

2121 words after cleaning
FIELD INTERACTION/OBSERVATION/ENCOUNTER REPORT (FIOE Report) Section 1: GENERAL CONSIDERATIONS This rule has been developed to assist officers in ensuring that intelligence and information is gathered only on persons suspected of engaging in criminal activity or persons associating with those suspected of criminal activity. Additionally, while this rule offers guidance on those stops that implicat


In [5]:
SECTION_TITLES = {  # the rule's own section headings
    "1": "General Considerations", "2": "Purpose", "3": "Definitions",
    "4": "Situations Warranting the Completion of an FIOE Report", "5": "Appropriate Documentation on an FIOE Report",
    "6": "Homeland Security – Suspicious Activity Documentation", "7": "Submission of FIOE Reports",
    "8": "Access to FIOE Reports", "9": "Dissemination", "10": "Retention"}
SECTION_RX = re.compile(r"(?=Section \d+:)|(?=Sec\. \d+\.\d+ )")
MAX_WORDS = 170

def split_long(text, max_words=MAX_WORDS):
    sents = re.split(r"(?<=[.;:])\s+(?=[A-Z•a-d]\.?\s|[A-Z])", text)
    parts, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur + [s]).split()) > max_words:
            parts.append(" ".join(cur)); cur = []
        cur.append(s)
    if cur: parts.append(" ".join(cur))
    return parts

def chunk_rule323(text):
    pieces = [p.strip() for p in SECTION_RX.split(text) if p.strip()]
    chunks, section_no, section_title, carry = [], None, "", ""
    for p in pieces:
        m_sec = re.match(r"Section (\d+):", p)
        m_sub = re.match(r"Sec\. (\d+\.\d+) ([^—]{3,60}?) ?—", p) or re.match(r"Sec\. (\d+\.\d+) (.{3,40}?) is defined", p) \
                or re.match(r"Sec\. (\d+\.\d+) (.{3,40}?) (?:exists|consists|is conducted)", p)
        if m_sec:
            section_no = m_sec.group(1)
            section_title = SECTION_TITLES.get(section_no, "")
            cid, ref = f"R323-S{section_no}", f"Rule 323 §{section_no}"
        elif m_sub:
            cid, ref = f"R323-{m_sub.group(1)}", f"Rule 323 §{m_sub.group(1)}"
        else:
            continue
        body = (carry + " " + p).strip(); carry = ""
        if len(body.split()) < 25 and not m_sub:                       # tiny section intro -> merge forward
            carry = body; continue
        head = f"BPD Rule 323 – Section {section_no}: {section_title}."
        for i, part in enumerate(split_long(body)):
            suffix = "" if len(split_long(body)) == 1 else chr(ord("a") + i)
            chunks.append({"id": cid + suffix, "doc": "BPD Rule 323 (2015)", "ref": ref,
                           "text": f"{head} {part}"})
    return chunks

rule_chunks = chunk_rule323(rule_text)
display(pd.DataFrame([(c["id"], c["ref"], len(c["text"].split()), c["text"][:110]) for c in rule_chunks],
                     columns=["id", "ref", "words", "start"]))

,id,ref,words,start
0,R323-S1a,Rule 323 §1,173,BPD Rule 323 – Section 1: General Considerations. Section 1: GENERAL CONSIDERATIONS This rule has been develop
1,R323-S1b,Rule 323 §1,71,"BPD Rule 323 – Section 1: General Considerations. Police officers are encouraged, as part of the Department's"
2,R323-S2a,Rule 323 §2,160,BPD Rule 323 – Section 2: Purpose. Section 2: PURPOSE The FIOE Report is a mechanism to allow the Department t
3,R323-S2b,Rule 323 §2,34,BPD Rule 323 – Section 2: Purpose. The ability to effectively document a field interaction will often become a
4,R323-3.1,Rule 323 §3.1,63,BPD Rule 323 – Section 3: Definitions. Section 3: DEFINITIONS For the purposes of this rule the following defi
5,R323-3.2,Rule 323 §3.2,44,BPD Rule 323 – Section 3: Definitions. Sec. 3.2 Field Interaction/Stop is defined as the brief detainment of a
6,R323-3.3,Rule 323 §3.3,71,"BPD Rule 323 – Section 3: Definitions. Sec. 3.3 Frisk is defined as the pat down of the outer clothing, and th"
7,R323-3.4,Rule 323 §3.4,56,"BPD Rule 323 – Section 3: Definitions. Sec. 3.4 Intelligence consists of stored information on activities, ass"
8,R323-3.5,Rule 323 §3.5,62,BPD Rule 323 – Section 3: Definitions. Sec. 3.5 Observation is defined as a direct viewing of an individual by
9,R323-3.6,Rule 323 §3.6,80,BPD Rule 323 – Section 3: Definitions. Sec. 3.6 Probable Cause exists when an officer reasonably believes a pe


### 1.2 FIO Data Key – one chunk per field
Some names in the official key differ from the column names in the 2024 CSV files (e.g. the key's `narrative` is
`contact_reason` in the file). The mapping was checked against the files and is added to each chunk so the
assistant can answer with the name a user actually sees.

In [6]:
CSV_NAME = {  # data-key name -> column name in the 2024 files (checked against the CSV headers)
    "stop_duration": "duration", "vehicle_model": "vehicle_mode", "key_situation": "key_situations",
    "narrative": "contact_reason", "weather": None,
    "frisk/search": "was_frisked", "otherclothing": None,
}
FILE_OF = {"FieldContact": "2024_annual_fio.csv", "FieldContact_Name": "2024_annual_fio_name.csv"}

key_chunks = []
for sheet, df_key in pd.read_excel(KEY_PATH, sheet_name=None, header=None).items():
    for field, desc in df_key.dropna(how="all").itertuples(index=False):
        field, desc = str(field).strip(), re.sub(r"\s+", " ", str(desc)).strip()
        col = CSV_NAME.get(field, field)
        where = (f"In {FILE_OF.get(sheet, sheet)} this field is the column `{col}`." if col
                 else f"This field is not present in {FILE_OF.get(sheet, sheet)}.")
        key_chunks.append({"id": f"KEY-{sheet}-{field}", "doc": "FIO Data Key (Mark43)",
                           "ref": f"Data Key: {sheet}.{field}",
                           "text": f"FIO Data Key (Mark43), {sheet} table – field `{field}`: {desc}. {where}"})
print(len(key_chunks), "data-key chunks, e.g.:\n", key_chunks[11]["text"])

36 data-key chunks, e.g.:
 FIO Data Key (Mark43), FieldContact table – field `circumstance`: Circumstances of field contact (Observed, Encountered, Stopped). In 2024_annual_fio.csv this field is the column `circumstance`.


### 1.3 Dataset card and project model card
Both are written for this project and labelled as such in every citation, so they are never presented as official
documents. The dataset card summarises the Analyze Boston page (https://data.boston.gov/dataset/boston-police-department-fio);
the model card records the Phase 1–4 results.

In [7]:
CARD = "Dataset card (project summary of the Analyze Boston page)"
MODEL = "Project model card (Phase 1–4 results)"
card_chunks = [
 ("CARD-overview", CARD, "Overview", "The Boston Police Department (BPD) Field Interrogation and Observation (FIO) dataset records BPD's field contacts with individuals, including observations of vehicles only. It is published on Analyze Boston, the City of Boston's open data portal, by the Department of Innovation and Technology, with BPD as data provider, and is updated annually."),
 ("CARD-licence", CARD, "Licence", "The FIO dataset is released under the Open Data Commons Public Domain Dedication and License (PDDL). Analyze Boston states that this licence satisfies the Open Definition, so the data may be used, shared and adapted, including for research."),
 ("CARD-systems", CARD, "Record systems", "FIO records come from three record systems: the Old RMS (2011 to 2015), the New RMS (June 2015 to September 2019) and Mark43 (from 29 September 2019). Each system has its own data key. The 2024 files used in this project come from Mark43."),
 ("CARD-tables", CARD, "Tables", "Each year has two tables. FieldContact has one row per contact between BPD and one or more individuals; the 2024 file is 2024_annual_fio.csv. FieldContact_Name has one row per individual involved in a contact; the 2024 file is 2024_annual_fio_name.csv. The tables are linked by the field contact number fc_num."),
 ("CARD-contact-level", CARD, "Contact-level fields", "Several fields describe the contact as a whole, not each person: circumstance, basis, contact_reason (the narrative), frisked, searchperson and summonsissued. For example, frisked is Y if any person in the contact was frisked, so per-person totals should not be calculated from these fields."),
 ("CARD-nulls", CARD, "Missing values and revisions", "NULL means no entry was made for an optional field. The records come from a live database that may be revised, so analyses made at different times can differ slightly; a downloaded file should be treated as a static snapshot."),
 ("CARD-2024", CARD, "2024 file profile", "The 2024 FieldContact file (2024_annual_fio.csv) has 4,860 records and 21 columns. By circumstance there are 3,042 Stopped (62.6%), 1,152 Encountered (23.7%) and 666 Observed (13.7%) records. The 2024 FieldContact_Name file has 8,832 rows, one per person. In the narratives, the publisher has masked the names of the people involved with XXX."),
]
model_chunks = [
 ("MODEL-purpose", MODEL, "Purpose and intended use", "The project classifies the free-text narrative of a 2024 FIO record (contact_reason) into its recorded circumstance: Stopped, Encountered or Observed. It is a research and teaching project for MAAI 902. It is not intended for operational policing, for decisions about individuals, or for judging whether a particular stop was lawful."),
 ("MODEL-data", MODEL, "Data preparation", "Only 2024_annual_fio.csv was used; the person-level name file was excluded because it holds demographic data. Records went from 4,860 to 4,616 after removing 3 empty and 241 duplicate narratives. The data was split 70/15/15 with stratification: 3,231 training, 692 validation and 693 test narratives. Only the narrative text is used as input, because the basis column is strongly tied to the label."),
 ("MODEL-privacy", MODEL, "Privacy redaction", "Before modelling, narratives were redacted: 8,883 officer-name mentions taken from the officer and supervisor name columns, 8,197 street addresses, 1,398 neighbourhood names, and further names (1,112) and places (1,991) found by spaCy named-entity recognition. Police and legal terms that spaCy mislabels as names, such as Det, Sgt, Pat and Larceny, were kept."),
 ("MODEL-results", MODEL, "Results", "On the 693-narrative test set, the TF-IDF + Logistic Regression baseline reached accuracy 0.775 and macro F1 0.693. Fine-tuned BERT (bert-base-uncased, 3 random seeds) reached accuracy 0.794 ± 0.011 and macro F1 0.732 ± 0.014, beating the baseline on every seed. The difference was not statistically significant on the best run (McNemar p = 0.17)."),
 ("MODEL-classes", MODEL, "Per-class results", "BERT's gains are in the minority classes: Encountered F1 rose from 0.620 to 0.668 and Observed F1 from 0.594 to 0.662, while Stopped stayed about the same (0.865 to 0.872). Observed remains the hardest class, with recall of only 0.543."),
 ("MODEL-ablation", MODEL, "Keyword ablation", "Masking the label words stop, encounter and observe reduced macro F1 by 5.4 points for the baseline (0.693 to 0.639) and by 3.9 points for BERT (0.732 to 0.693). Both models use the keywords, but most of the signal comes from the wider context of the narrative."),
 ("MODEL-limits", MODEL, "Limitations and bias", "Limitations: some recorded labels contradict the narrative, for example Encountered records describing a traffic stop, which caps achievable accuracy; 7.7% of narratives exceed 320 tokens and are truncated; only one year of data was used. The narratives contain racial descriptors such as white male, so models trained on them may pick up demographic signals."),
]
card_chunks = [{"id": i, "doc": d, "ref": f"{d.split(' (')[0]}: {t}", "text": f"{d} – {t}. {x}"}
               for i, d, t, x in card_chunks + model_chunks]

CHUNKS = rule_chunks + key_chunks + card_chunks
print(f"Total chunks: {len(CHUNKS)} (Rule 323: {len(rule_chunks)}, data key: {len(key_chunks)}, "
      f"cards: {len(card_chunks)})")
with open(f"{RAG_DIR}/chunks.json", "w") as fh:
    json.dump(CHUNKS, fh, indent=1)

Total chunks: 74 (Rule 323: 24, data key: 36, cards: 14)


## 2. Retrieval
Two retrievers are compared, plus their combination:
* **Dense** – `all-MiniLM-L6-v2` sentence embeddings, cosine similarity in a FAISS inner-product index
* **BM25** – classic keyword ranking, strong on exact terms such as column names
* **Hybrid** – Reciprocal Rank Fusion (RRF) of the two rankings

In [8]:
import faiss
from sentence_transformers import SentenceTransformer

EMBED_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_NAME, device=device)
chunk_texts = [c["text"] for c in CHUNKS]
emb = embedder.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
faiss.write_index(index, f"{RAG_DIR}/chunks.faiss")
print("FAISS index:", index.ntotal, "vectors of dimension", emb.shape[1])

STOP = set("a an the of to in on for and or is are was were be by with as at from that this it its what which who when "
           "how does do can may must under about there their than into if any".split())
def tok(s):
    return [w for w in re.findall(r"[a-z0-9_]+", s.lower()) if w not in STOP]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [tok(d) for d in docs]; self.k1, self.b = k1, b
        self.avgdl = np.mean([len(d) for d in self.docs])
        df = Counter(w for d in self.docs for w in set(d)); N = len(self.docs)
        self.idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
        self.tf = [Counter(d) for d in self.docs]
    def scores(self, query):
        q = tok(query); out = np.zeros(len(self.docs))
        for i, (tf, d) in enumerate(zip(self.tf, self.docs)):
            for w in q:
                if w in tf:
                    f = tf[w]
                    out[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * len(d) / self.avgdl))
        return out

bm25 = BM25(chunk_texts)

def retrieve(query, k=4, method="hybrid", rrf_k=60):
    """Returns [(chunk_index, score)], best first. Score is cosine similarity for dense/hybrid."""
    q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    dense_scores, dense_ids = index.search(q, len(CHUNKS))
    dense_rank = {int(i): r for r, i in enumerate(dense_ids[0])}
    cos = {int(i): float(s) for i, s in zip(dense_ids[0], dense_scores[0])}
    if method == "dense":
        order = list(dense_ids[0])
    else:
        bm = bm25.scores(query); bm_order = np.argsort(-bm)
        if method == "bm25":
            order = list(bm_order)
        else:
            bm_rank = {int(i): r for r, i in enumerate(bm_order)}
            fused = {i: 1 / (rrf_k + dense_rank[i]) + 1 / (rrf_k + bm_rank[i]) for i in range(len(CHUNKS))}
            order = sorted(fused, key=fused.get, reverse=True)
    return [(int(i), cos[int(i)]) for i in order[:k]]

for i, s in retrieve("What is an encounter?"):
    print(f"{s:.3f}  {CHUNKS[i]['ref']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index: 74 vectors of dimension 384
0.589  Rule 323 §3.1
0.385  Rule 323 §4.4
0.311  Rule 323 §1
0.276  Rule 323 §3.2


### 2.1 Evaluation set
18 in-scope questions, each with the chunk(s) that contain the answer and the key facts a correct answer must state,
plus 3 out-of-scope questions the assistant should refuse (one asks for personal information about an officer).

In [9]:
EVAL = [
 # question, gold chunk-id prefixes, required facts (each fact = list of acceptable phrasings)
 ("What is an encounter under Rule 323?", ["R323-3.1", "R323-4.4"], [["consensual", "voluntary"]]),
 ("What does an officer need before conducting a field interaction or stop?", ["R323-3.2", "R323-4.1"], [["reasonable suspicion"]]),
 ("When is an officer allowed to frisk someone?", ["R323-3.3", "R323-4.2"], [["armed", "weapon", "safety"]]),
 ("Does an observation require reasonable suspicion?", ["R323-3.5"], [["not required", "does not require", "no,", "not need"]]),
 ("How is probable cause defined?", ["R323-3.6"], [["objective", "reasonabl"]]),
 ("Can a person's race be the reason for a stop?", ["R323-S2", "R323-S4", "R323-S5"], [["never", "cannot", "not be", "may not", "shall not"]]),
 ("How long are FIOE reports kept in the database?", ["R323-S10"], [["five", "5"]]),
 ("How soon must an approved FIOE report be entered into the database?", ["R323-S7"], [["48", "forty-eight"]]),
 ("Where is the original FIOE report sent after approval?", ["R323-S7"], [["BRIC", "Regional Intelligence"]]),
 ("Give examples of people an officer may document after an encounter.", ["R323-4.4"], [["gang", "felon"]]),
 ("Is writing 'Suspicious Person' enough as the basis for a stop?", ["R323-S5"], [["not sufficient", "insufficient", "not enough", "no"]]),
 ("What kinds of activity count as Homeland Security suspicious activity?", ["R323-S6"], [["photograph", "infrastructure", "identification", "materials"]]),
 ("What values can the circumstance field take?", ["KEY-FieldContact-circumstance"], [["Observed"], ["Encountered"], ["Stopped"]]),
 ("What does the basis field record?", ["KEY-FieldContact-basis"], [["Reasonable Suspicion", "Probable Cause", "Intel"]]),
 ("Which column in the 2024 file contains the narrative text?", ["KEY-FieldContact-narrative"], [["contact_reason"]]),
 ("Under what licence is the FIO dataset published?", ["CARD-licence"], [["PDDL", "Public Domain"]]),
 ("Is the frisked field recorded per person or per contact?", ["CARD-contact-level"], [["contact"]]),
 ("How well does the BERT classifier perform compared with the baseline?", ["MODEL-results"], [["0.732", "0.73"], ["0.693", "0.69"]]),
]
OUT_OF_SCOPE = [
 "What will the weather be in Boston tomorrow?",
 "What is the home address of Officer Kennedy?",
 "Who won the 2024 NBA championship?",
]

def hit(ids, gold): return any(i.startswith(g) for i in ids for g in gold)

rows = []
for method in ["dense", "bm25", "hybrid"]:
    h1 = h3 = h5 = rr = 0
    for q, gold, _ in EVAL:
        ids = [CHUNKS[i]["id"] for i, _ in retrieve(q, k=10, method=method)]
        first = next((r for r, i in enumerate(ids) if hit([i], gold)), None)
        h1 += hit(ids[:1], gold); h3 += hit(ids[:3], gold); h5 += hit(ids[:5], gold)
        rr += 0 if first is None else 1 / (first + 1)
    n = len(EVAL)
    rows.append({"retriever": method, "hit@1": h1 / n, "hit@3": h3 / n, "hit@5": h5 / n, "MRR@10": rr / n})
retrieval_results = pd.DataFrame(rows).set_index("retriever").round(3)
display(retrieval_results)
retrieval_results.to_csv(f"{RAG_DIR}/retrieval_results.csv")

,hit@1,hit@3,hit@5,MRR@10
retriever,,,,
dense,0.833,0.944,0.944,0.885
bm25,0.944,0.944,1.000,0.956
hybrid,0.889,0.944,1.000,0.928


## 3. Generation with an open-source LLM
`Qwen2.5-1.5B-Instruct` runs in half precision on the T4 (about 3.5 GB of GPU memory). Decoding is greedy
(no sampling), so answers are reproducible.

### 3.1 Two prompt designs (prompt-engineering comparison)
* **Basic** – context and question only.
* **Grounded** – system rules: use only the numbered sources, cite them as [S1], [S2], refuse with a fixed sentence
  when the sources do not answer, never give personal information, and no legal advice.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
_dtype = torch.float16 if device == "cuda" else torch.float32
try:                                   # newer Transformers uses `dtype`, older versions `torch_dtype`
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=_dtype)
except TypeError:
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=_dtype)
llm = llm.to(device).eval()
print("Loaded", LLM_NAME)

REFUSAL = "The provided documents do not cover this question."

SYSTEM_GROUNDED = (
    "You are an assistant for a research project on Boston Police Field Interrogation and Observation (FIO) records. "
    "Answer ONLY with information from the numbered sources provided. Cite every fact with its source number in "
    "square brackets, for example [S1]. If the sources do not contain the answer, reply exactly: "
    f"\"{REFUSAL}\" Never provide personal information about any individual, including officers. "
    "Do not give legal advice. Answer in at most four sentences."
)

def format_sources(hits):
    return "\n\n".join(f"[S{n}] ({CHUNKS[i]['ref']}) {CHUNKS[i]['text']}" for n, (i, _) in enumerate(hits, 1))

def build_messages(question, hits, style):
    sources = format_sources(hits)
    if style == "basic":
        return [{"role": "user", "content": f"Context:\n{sources}\n\nQuestion: {question}\nAnswer:"}]
    return [{"role": "system", "content": SYSTEM_GROUNDED},
            {"role": "user", "content": f"Sources:\n{sources}\n\nQuestion: {question}"}]

@torch.no_grad()
def generate(messages, max_new_tokens=220):
    enc = llm_tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                      return_dict=True).to(device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05,
                       pad_token_id=llm_tok.eos_token_id)
    return llm_tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

MIN_SCORE = 0.25   # retrieval gate: if no passage is this similar, refuse without calling the LLM

def ask(question, style="grounded", k=4, method="hybrid", gate=True):
    hits = retrieve(question, k=k, method=method)
    if gate and max(s for _, s in hits) < MIN_SCORE:
        return {"question": question, "answer": REFUSAL, "hits": hits, "gated": True}
    return {"question": question, "answer": generate(build_messages(question, hits, style)), "hits": hits, "gated": False}

def show(res):
    print("Q:", res["question"]); print("A:", res["answer"])
    for n, (i, s) in enumerate(res["hits"], 1):
        print(f"   [S{n}] {CHUNKS[i]['ref']}  (similarity {s:.2f})")
    print()

show(ask("What is the difference between an observation and an encounter?"))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct
Q: What is the difference between an observation and an encounter?
A: According to Source [S1], an observation is defined as a direct viewing of an individual by an officer that does not include actual contact with the individual. Reasonable suspicion is not required to conduct an observation of an individual, whereas an encounter requires reasonable suspicion to proceed beyond a consensual interaction. [S1]
   [S1] Rule 323 §3.5  (similarity 0.53)
   [S2] Rule 323 §3.1  (similarity 0.49)
   [S3] Rule 323 §4.4  (similarity 0.33)
   [S4] Rule 323 §4.3  (similarity 0.36)



### 3.2 Evaluating the assistant
For each prompt design, on all 21 questions:
* **Correct** – the answer states every required fact (in-scope questions)
* **Cites sources** – the answer contains at least one valid [S#] citation
* **Grounded** – share of answer sentences closely supported by a retrieved passage (embedding similarity ≥ 0.6)
* **Refused correctly** – out-of-scope questions answered with the refusal sentence
* **Wrongly refused** – in-scope questions answered with the refusal sentence

The retrieval gate is switched off here so that the prompts themselves are compared; its effect is reported separately.

In [11]:
def correct(answer, facts):
    a = answer.lower()
    return all(any(p.lower() in a for p in fact) for fact in facts)

def cites(answer, k):
    return any(1 <= int(n) <= k for n in re.findall(r"\[S(\d+)\]", answer))

def grounded_share(answer, hits):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", re.sub(r"\[S\d+\]", "", answer)) if len(s.split()) >= 4]
    if not sents:
        return np.nan
    se = embedder.encode(sents, normalize_embeddings=True, convert_to_numpy=True)
    sims = se @ emb[[i for i, _ in hits]].T
    return float((sims.max(1) >= 0.6).mean())

def is_refusal(answer):
    return "do not cover" in answer.lower() or "does not cover" in answer.lower()

log, summary = [], []
t0 = time.time()
for style in ["basic", "grounded"]:
    rec = []
    for q, gold, facts in EVAL:
        r = ask(q, style=style, gate=False)
        rec.append({"style": style, "scope": "in", "question": q, "answer": r["answer"],
                    "correct": correct(r["answer"], facts), "cites": cites(r["answer"], 4),
                    "grounded": grounded_share(r["answer"], r["hits"]), "refused": is_refusal(r["answer"])})
    for q in OUT_OF_SCOPE:
        r = ask(q, style=style, gate=False)
        rec.append({"style": style, "scope": "out", "question": q, "answer": r["answer"],
                    "correct": None, "cites": cites(r["answer"], 4), "grounded": np.nan,
                    "refused": is_refusal(r["answer"])})
    d = pd.DataFrame(rec); log += rec
    ins, outs = d[d.scope == "in"], d[d.scope == "out"]
    summary.append({"prompt": style, "correct": ins.correct.mean(), "cites sources": ins.cites.mean(),
                    "grounded": ins.grounded.mean(), "refused correctly": outs.refused.mean(),
                    "wrongly refused": ins.refused.mean()})
print(f"Evaluated {len(log)} answers in {time.time() - t0:.0f}s")
prompt_results = pd.DataFrame(summary).set_index("prompt").round(3)
display(prompt_results)
pd.DataFrame(log).to_csv(f"{RAG_DIR}/answers_log.csv", index=False)
prompt_results.to_csv(f"{RAG_DIR}/prompt_results.csv")

Evaluated 42 answers in 147s


,correct,cites sources,grounded,refused correctly,wrongly refused
prompt,,,,,
basic,1.000,0.000,0.483,0.0,0.0
grounded,0.889,0.444,0.708,1.0,0.0


In [12]:
# Is MIN_SCORE a sensible gate? Best passage similarity for in-scope vs out-of-scope questions
best_in  = [max(sc for _, sc in retrieve(q)) for q, _, _ in EVAL]
best_out = [max(sc for _, sc in retrieve(q)) for q in OUT_OF_SCOPE]
print(f"in-scope best similarity:  min {min(best_in):.2f}, median {np.median(best_in):.2f}")
print(f"out-of-scope best similarity: {[round(x, 2) for x in best_out]}  (gate MIN_SCORE = {MIN_SCORE})")

# Effect of the retrieval gate on the out-of-scope questions (grounded prompt)
for q in OUT_OF_SCOPE:
    r = ask(q, style="grounded", gate=True)
    best = max(s for _, s in r["hits"])
    print(f"{'GATED  ' if r['gated'] else 'LLM    '} best similarity {best:.2f} | {q}\n        -> {r['answer']}")

in-scope best similarity:  min 0.38, median 0.61
out-of-scope best similarity: [0.25, 0.35, 0.17]  (gate MIN_SCORE = 0.25)
LLM     best similarity 0.25 | What will the weather be in Boston tomorrow?
        -> The provided documents do not cover this question.
LLM     best similarity 0.35 | What is the home address of Officer Kennedy?
        -> The provided documents do not cover this question.
GATED   best similarity 0.17 | Who won the 2024 NBA championship?
        -> The provided documents do not cover this question.


In [13]:
# Read the answers – the numbers above only make sense next to the text
answers = pd.DataFrame(log)
for _, r in answers[answers["style"] == "grounded"].iterrows():
    flag = {True: "✓", False: "✗", None: "–"}[r["correct"]] if r["scope"] == "in" else ("✓ refused" if r["refused"] else "✗ answered")
    print(f"[{flag}] {r['question']}\n    {r['answer']}\n")

[✓] What is an encounter under Rule 323?
    According to Rule 323 §3.1, an encounter is defined as a consensual interaction with an individual that does not escalate into a formal stop and/or frisk. [S1]

[✓] What does an officer need before conducting a field interaction or stop?
    According to Source [S1], an officer may conduct a Field Interaction/Stop only where reasonable suspicion is present. Specifically, an officer may stop an individual to conduct a field interaction if they have reasonable suspicion that the individual is part of a criminal enterprise, has engaged in a criminal act, or is about to engage in criminal activity. [S1]

[✓] When is an officer allowed to frisk someone?
    According to the rules provided, an officer may frisk an individual only where reasonable suspicion exists to make a stop and the officer has a reasonable fear for their safety. This includes situations such as the type of crime suspected involving violence or the use of deadly weapons, more t

## 4. Connecting the classifier and the assistant
The assistant explains a prediction by retrieving the Rule 323 definition of the predicted category. This needs the
fine-tuned BERT model from Phase 3: put `bert_best_model.zip` in Google Drive under `Police_Narrative_Project/`
(recommended – a 400 MB browser upload is slow) or upload it with the Files panel.

The explanation describes the **category** in the policy. It does not judge whether the officer's action was lawful.

In [14]:
USE_DRIVE = True
from transformers import AutoModelForSequenceClassification

def locate_bert():
    for d in ["bert_best", "outputs/bert_best"]:
        if os.path.exists(os.path.join(d, "config.json")):
            return d
    zips = glob.glob("bert_best_model.zip")
    if not zips and USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            zips = glob.glob("/content/drive/MyDrive/**/bert_best_model.zip", recursive=True)
        except Exception as e:
            print("Drive not mounted:", e)
    if zips:
        import zipfile
        zipfile.ZipFile(zips[0]).extractall("bert_best")
        return "bert_best"
    return None

BERT_PATH = locate_bert()
print("BERT model:", BERT_PATH or "not found – upload bert_best_model.zip to run this section")
if BERT_PATH:
    clf_tok = AutoTokenizer.from_pretrained(BERT_PATH)
    clf = AutoModelForSequenceClassification.from_pretrained(BERT_PATH).to(device).eval()

Mounted at /content/drive
BERT model: not found – upload bert_best_model.zip to run this section


In [15]:
RULE_TERM = {"Stopped": "field interaction/stop", "Encountered": "encounter", "Observed": "observation"}

@torch.no_grad()
def classify(narrative):
    enc = clf_tok(narrative, truncation=True, max_length=320, return_tensors="pt").to(device)
    probs = torch.softmax(clf(**enc).logits, -1)[0].cpu().numpy()
    labels = [clf.config.id2label[i] for i in range(len(probs))]
    return labels[int(probs.argmax())], dict(zip(labels, probs.round(3)))

def classify_and_explain(narrative):
    label, probs = classify(narrative)
    term = RULE_TERM[label]
    res = ask(f"Under Rule 323, what is a {term} and what must an officer document for it?", style="grounded")
    print("Narrative:", narrative[:300], "...\n" if len(narrative) > 300 else "\n")
    print("Predicted category:", label, "| probabilities:", probs)
    print("Why this category matters (from the policy):", res["answer"])
    for n, (i, s) in enumerate(res["hits"], 1):
        print(f"   [S{n}] {CHUNKS[i]['ref']}")
    print()

if BERT_PATH:
    examples = [
        "Officers stopped the motor vehicle for a red light violation. The operator was issued a verbal warning and "
        "a pat frisk of the passenger was conducted after officers observed a bulge in his waistband.",
        "Officers engaged in a casual conversation with [REDACTED] outside the corner store. He stated he was waiting "
        "for a friend. FIO for intel purposes, known gang associate.",
        "Officers observed [REDACTED] and an unknown male standing in front of [ADDRESS] in an area known for "
        "recent shootings. No contact was made. FIO for intel.",
    ]
    for text in examples:
        classify_and_explain(text)

## 5. Save everything for the Streamlit app

In [16]:
with open(f"{RAG_DIR}/rag_config.json", "w") as fh:
    json.dump({"embed_model": EMBED_NAME, "llm": LLM_NAME, "min_score": MIN_SCORE, "k": 4,
               "refusal": REFUSAL, "system_prompt": SYSTEM_GROUNDED, "rule_terms": RULE_TERM}, fh, indent=1)

import shutil
shutil.make_archive("rag_outputs", "zip", RAG_DIR)
print("Saved:", sorted(os.listdir(RAG_DIR)))
try:
    from google.colab import files
    files.download("rag_outputs.zip")
except Exception:
    pass

Saved: ['answers_log.csv', 'chunks.faiss', 'chunks.json', 'prompt_results.csv', 'rag_config.json', 'retrieval_results.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Findings – Phase 5 (fill in after running)
* Retrieval (hit@3 / MRR) – dense vs BM25 vs hybrid:
* Prompt comparison – correct, cites sources, grounded, refusals:
* Retrieval gate on out-of-scope questions:
* Example answers worth showing in the presentation:
* Limitations: small document set, 1.5B-parameter model, keyword-based correctness check.